# Métricas de classificação de sentimento (-1, 0, +1)

Versão para **Jupyter local** (não depende do Google Colab).

Você só precisa preencher o caminho do CSV na célula de configuração — o resto roda automaticamente.

## 1. Importações

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROTULOS_PADRAO = [-1, 0, 1]
NOMES = {-1: "Negativo (-1)", 0: "Neutro (0)", 1: "Positivo (+1)"}

# Variantes textuais aceitas, caso o CSV traga palavras em vez de numeros.
MAPA_TEXTO = {
    "neg": -1, "negativo": -1, "negative": -1,
    "neu": 0, "neutro": 0, "neutral": 0,
    "pos": 1, "positivo": 1, "positive": 1,
}

## 2. Funções (idênticas ao script original)

In [ ]:
# --------------------------------------------------------------------------- #
# Leitura e limpeza dos dados
# --------------------------------------------------------------------------- #
def ler_csv(caminho):
    """Le o CSV detectando o separador (virgula, ponto e virgula, tab...).

    Tenta primeiro utf-8-sig, depois latin-1. Cada tentativa pode falhar por
    problema de decodificacao (UnicodeDecodeError) ou por o parser nao
    conseguir inferir a estrutura do arquivo (ParserError) -- ambos sao
    tratados aqui para nao mascarar o erro real com uma mensagem generica.
    """
    caminho = Path(caminho).expanduser()
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo nao encontrado: {caminho}\n"
            f"Confira o caminho exato com: ls \"{caminho.parent}\""
        )

    erros = []
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(caminho, sep=None, engine="python", encoding=enc)
        except (UnicodeDecodeError, pd.errors.ParserError) as e:
            erros.append(f"{enc}: {e}")
            continue
        except pd.errors.EmptyDataError:
            raise SystemExit(f"Arquivo CSV vazio: {caminho}")

    detalhe = "\n  - ".join(erros)
    raise SystemExit(
        "Nao foi possivel ler o arquivo CSV com nenhuma das codificacoes testadas "
        f"(utf-8-sig, latin-1).\nDetalhes:\n  - {detalhe}"
    )


def normalizar_rotulos(serie):
    """Converte '+1', '1.0', 'positivo', etc. para numeros; invalidos viram NaN."""
    s = serie.astype(str).str.strip().str.lower().str.replace(",", ".", regex=False)
    s = s.replace(MAPA_TEXTO)
    return pd.to_numeric(s, errors="coerce")


def preparar_dados(df, col_pred, col_gold, rotulos, cols_prob=None):
    """Retorna (y_pred, y_gold, probas) como arrays, com linhas invalidas removidas."""
    col_pred = col_pred or df.columns[0]
    col_gold = col_gold or df.columns[1]
    for c in (col_pred, col_gold, *(cols_prob or [])):
        if c not in df.columns:
            raise SystemExit(f"Coluna '{c}' nao encontrada. Colunas disponiveis: {list(df.columns)}")

    pred = normalizar_rotulos(df[col_pred])
    gold = normalizar_rotulos(df[col_gold])
    validos = pred.notna() & gold.notna()

    probas = None
    if cols_prob:
        if len(cols_prob) != len(rotulos):
            raise SystemExit(f"prob_cols precisa de {len(rotulos)} colunas (uma por classe, na ordem {rotulos}).")
        probas = df[cols_prob].apply(pd.to_numeric, errors="coerce")
        validos &= probas.notna().all(axis=1)

    fora = validos & (~pred.isin(rotulos) | ~gold.isin(rotulos))
    n_nan = int((~validos).sum())
    n_fora = int(fora.sum())
    if n_nan:
        print(f"[aviso] {n_nan} linha(s) ignorada(s) por valor ausente ou nao numerico.")
    if n_fora:
        print(f"[aviso] {n_fora} linha(s) ignorada(s) por rotulo fora de {rotulos}.")
    validos &= ~fora

    if validos.sum() == 0:
        raise SystemExit("Nenhuma linha valida para avaliar.")

    y_pred = pred[validos].astype(int).to_numpy()
    y_gold = gold[validos].astype(int).to_numpy()
    p = probas[validos].to_numpy(dtype=float) if probas is not None else None
    return y_pred, y_gold, p


# --------------------------------------------------------------------------- #
# Metricas
# --------------------------------------------------------------------------- #
def dividir(a, b):
    """Divisao segura: retorna 0.0 quando o denominador e zero."""
    return float(a) / float(b) if b else 0.0


def matriz_confusao(y_gold, y_pred, rotulos):
    """Linhas = classe real (gold); colunas = classe predita pela IA."""
    idx = {r: i for i, r in enumerate(rotulos)}
    k = len(rotulos)
    cm = np.zeros((k, k), dtype=int)
    np.add.at(cm, ([idx[g] for g in y_gold], [idx[p] for p in y_pred]), 1)
    return cm


def metricas_por_classe(cm, rotulos):
    """Precisao, revocacao, especificidade e F1 por classe (um-contra-todos)."""
    total = cm.sum()
    linhas = []
    for i, r in enumerate(rotulos):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        prec = dividir(tp, tp + fp)
        rec = dividir(tp, tp + fn)
        esp = dividir(tn, tn + fp)
        f1 = dividir(2 * prec * rec, prec + rec)
        linhas.append({
            "Classe": NOMES.get(r, str(r)),
            "Precisao": prec, "Revocacao": rec, "Especificidade": esp, "F1": f1,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn, "Suporte": int(cm[i, :].sum()),
        })
    return pd.DataFrame(linhas)


def mcc_multiclasse(cm):
    """MCC multiclasse (Gorodkin) a partir da matriz de confusao."""
    t = cm.sum(axis=1).astype(float)
    p = cm.sum(axis=0).astype(float)
    c = float(np.trace(cm))
    s = float(cm.sum())
    num = c * s - float(np.dot(t, p))
    den = np.sqrt((s ** 2 - float(np.dot(p, p))) * (s ** 2 - float(np.dot(t, t))))
    return dividir(num, den)


def auc_binario(y_bin, score):
    """AUC pela estatistica de Mann-Whitney (trata empates com postos medios)."""
    y_bin = np.asarray(y_bin).astype(int)
    n_pos = int(y_bin.sum())
    n_neg = len(y_bin) - n_pos
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    postos = pd.Series(np.asarray(score, dtype=float)).rank(method="average").to_numpy()
    return (postos[y_bin == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def roc_auc_ovr(y_gold, y_pred, rotulos, probas=None):
    """AUC um-contra-todos por classe e media macro."""
    aucs = {}
    for i, r in enumerate(rotulos):
        y_bin = (y_gold == r).astype(int)
        score = probas[:, i] if probas is not None else (y_pred == r).astype(float)
        aucs[r] = auc_binario(y_bin, score)
    macro = float(np.nanmean(list(aucs.values()))) if not all(np.isnan(list(aucs.values()))) else float("nan")
    return aucs, macro


def calcular_tudo(y_gold, y_pred, rotulos, probas=None):
    cm = matriz_confusao(y_gold, y_pred, rotulos)
    por_classe = metricas_por_classe(cm, rotulos)
    suporte = por_classe["Suporte"].to_numpy()

    macro_f1 = por_classe["F1"].mean()
    f1_pond = float(np.average(por_classe["F1"], weights=suporte)) if suporte.sum() else 0.0
    macro_rec = por_classe["Revocacao"].mean()

    aucs, auc_macro = roc_auc_ovr(y_gold, y_pred, rotulos, probas)
    por_classe["ROC-AUC (OvR)"] = [aucs[r] for r in rotulos]

    globais = {
        "N amostras": int(cm.sum()),
        "Acuracia": dividir(np.trace(cm), cm.sum()),
        "Precisao (macro)": por_classe["Precisao"].mean(),
        "Revocacao (macro)": macro_rec,
        "Especificidade (macro)": por_classe["Especificidade"].mean(),
        "F1 (macro)": macro_f1,
        "F1 (ponderado)": f1_pond,
        "Balanced Accuracy": macro_rec,
        "MCC": mcc_multiclasse(cm),
        "ROC-AUC (OvR, macro)": auc_macro,
    }
    return cm, por_classe, globais


# --------------------------------------------------------------------------- #
# Saidas
# --------------------------------------------------------------------------- #
def salvar_figura(cm, rotulos, caminho):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("[aviso] matplotlib nao instalado; figura da matriz de confusao nao gerada.")
        return False

    nomes = [NOMES.get(r, str(r)) for r in rotulos]
    norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, eixos = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, dados, titulo, fmt in (
        (eixos[0], cm, "Matriz de confusao (contagens)", "d"),
        (eixos[1], norm, "Normalizada por linha (revocacao)", ".2f"),
    ):
        im = ax.imshow(dados, cmap="Blues")
        ax.set_title(titulo, fontsize=11)
        ax.set_xticks(range(len(nomes)), nomes, rotation=20)
        ax.set_yticks(range(len(nomes)), nomes)
        ax.set_xlabel("Predito pela IA")
        ax.set_ylabel("Real (padrao-ouro)")
        limiar = dados.max() / 2 if dados.max() else 0
        for i in range(dados.shape[0]):
            for j in range(dados.shape[1]):
                ax.text(j, i, format(dados[i, j], fmt), ha="center", va="center",
                        color="white" if dados[i, j] > limiar else "black")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(caminho, dpi=200)
    plt.close(fig)
    return True


print("Funcoes carregadas.")

## 3. Configuração

Preencha `CAMINHO_CSV` com o caminho completo do seu arquivo. Dica: no terminal, rode `ls "/caminho/da/pasta"` para confirmar o nome exato (atenção a espaços, parênteses como `(1)`, acentos).

In [ ]:
CAMINHO_CSV = "/home/guest/Downloads/2026/giselda/Teste_metricas/saida_gpt_5.csv"  # <-- ajuste aqui

COL_PRED = None       # ex: "IA"          (None = usa a 1a coluna do CSV)
COL_GOLD = None       # ex: "Gold"        (None = usa a 2a coluna do CSV)
ROTULOS = [-1, 0, 1]  # classes, em ordem crescente
PROB_COLS = None      # ex: ["prob_neg", "prob_neu", "prob_pos"] (None = sem probabilidades)
PASTA_SAIDA = "resultados_metricas"

## 4. Calcular e exibir as métricas

In [ ]:
df = ler_csv(CAMINHO_CSV)
if df.shape[1] < 2:
    raise SystemExit("O CSV precisa ter pelo menos duas colunas (IA e padrao-ouro).")

y_pred, y_gold, probas = preparar_dados(df, COL_PRED, COL_GOLD, ROTULOS, PROB_COLS)
cm, por_classe, globais = calcular_tudo(y_gold, y_pred, ROTULOS, probas)

pd.options.display.float_format = "{:.4f}".format
pd.options.display.width = 140
nomes = [NOMES.get(r, str(r)) for r in ROTULOS]
cm_df = pd.DataFrame(cm, index=[f"Real {n}" for n in nomes], columns=[f"Pred {n}" for n in nomes])
glob_df = pd.DataFrame({"Metrica": list(globais.keys()), "Valor": list(globais.values())})

print("\n=== MATRIZ DE CONFUSAO (linhas = real, colunas = predito) ===")
print(cm_df.to_string())
print("\n=== METRICAS POR CLASSE (um-contra-todos) ===")
print(por_classe.drop(columns=["TP", "FP", "FN", "TN"]).to_string(index=False))
print("\n=== METRICAS GLOBAIS ===")
exibir = glob_df.copy()
exibir["Valor"] = [str(int(v)) if m == "N amostras" else f"{v:.4f}"
                   for m, v in zip(glob_df["Metrica"], glob_df["Valor"])]
print(exibir.to_string(index=False))
if probas is None:
    print("\nObs.: sem probabilidades no CSV, o ROC-AUC foi calculado com as predicoes duras; "
          "para uma curva ROC completa, defina PROB_COLS.")
print("Obs.: divisoes com denominador zero retornam 0.")

## 5. Ver a matriz de confusão (gráfico)

In [ ]:
saida = Path(PASTA_SAIDA)
saida.mkdir(parents=True, exist_ok=True)

figura_ok = salvar_figura(cm, ROTULOS, saida / "matriz_confusao.png")
if figura_ok:
    from IPython.display import Image, display
    display(Image(filename=str(saida / "matriz_confusao.png")))

## 6. Salvar os resultados (CSVs + gráfico)

Os arquivos ficam salvos na pasta `PASTA_SAIDA`, dentro do diretório onde o Jupyter foi iniciado. O caminho completo aparece impresso abaixo.

In [ ]:
cm_df.to_csv(saida / "matriz_confusao.csv", encoding="utf-8-sig")
por_classe.to_csv(saida / "metricas_por_classe.csv", index=False, encoding="utf-8-sig")
glob_df.to_csv(saida / "metricas_globais.csv", index=False, encoding="utf-8-sig")

print(f"Arquivos salvos em: {saida.resolve()}")